In [ ]:
from pathlib import Path
import zipfile
import requests
import geopandas as gpd
import shutil
from datetime import datetime

In [ ]:
basedir  = Path('/data/common/geometry/attribute_overlay_polygons/')
basedir.mkdir(exist_ok=True)

## Get RFC boundaries

In [ ]:
fetch_date = datetime.now().strftime("%m-%d-%Y")
url = "https://www.weather.gov/source/gis/Shapefiles/Misc/rf05mr24.zip"
parquet_path = Path(basedir, "rfc_geometry.all.parquet")

extract_dir = Path(basedir, "tmp_extract")
zip_path = Path(basedir, "temp.zip")

# Download zip
resp = requests.get(url, timeout=60)
resp.raise_for_status()
zip_path.write_bytes(resp.content)

# Extract
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(extract_dir)

# Find shapefile
shp = next(extract_dir.rglob("*.shp"))

# Read and convert
gdf = gpd.read_file(shp)
gdf = gdf.rename(columns = {'BASIN_ID':'id', 'RFC_NAME':'name'})
source_dict = {
    "source" : "NWS",
    "url" : url,
    "date_retreived" : {fetch_date},
}
gdf['properties'] = [source_dict] * len(gdf)
gdf[['id','name','geometry','properties']].to_parquet(parquet_path, index=False)

print(f"Saved GeoParquet: {parquet_path}")

# Cleanup: remove zip + extracted files
if zip_path.exists():
    zip_path.unlink()
if extract_dir.exists():
    shutil.rmtree(extract_dir)

In [ ]:
check = gpd.read_parquet(parquet_path)
check

## Get WFO boundaries

In [ ]:
fetch_date = datetime.now().strftime("%m-%d-%Y")
url = "https://www.weather.gov/source/gis/Shapefiles/WSOM/w_16ap26.zip"
parquet_path = Path(basedir, "wfo_geometry.all.parquet")

extract_dir = Path(basedir, "tmp_extract")
zip_path = Path(basedir, "temp.zip")

# Download zip
resp = requests.get(url, timeout=60)
resp.raise_for_status()
zip_path.write_bytes(resp.content)

# Extract
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(extract_dir)

# Find shapefile
shp = next(extract_dir.rglob("*.shp"))

# Read and convert
gdf = gpd.read_file(shp)

gdf = gdf.rename(columns = {'WFO':'id', 'CITYSTATE':'name'})

source_dict = {
    "source" : "NWS",
    "url" : url,
    "date_retreived" : {fetch_date},
}
gdf['properties'] = [source_dict] * len(gdf)
gdf[['id','name','geometry','properties']].to_parquet(parquet_path, index=False)

print(f"Saved GeoParquet: {parquet_path}")

# Cleanup: remove zip + extracted files
if zip_path.exists():
    zip_path.unlink()
if extract_dir.exists():
    shutil.rmtree(extract_dir)

In [ ]:
check = gpd.read_parquet(parquet_path)
check

## Get NWS Region Boundaries

In [ ]:
fetch_date = datetime.now().strftime("%m-%d-%Y")
url = "https://www.weather.gov/source/gis/Shapefiles/Misc/re15au11.zip"
parquet_path = Path(basedir, "nws_region_geometry.all.parquet")

extract_dir = Path(basedir, "tmp_extract")
zip_path = Path(basedir, "temp.zip")

# Download zip
resp = requests.get(url, timeout=60)
resp.raise_for_status()
zip_path.write_bytes(resp.content)

# Extract
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(extract_dir)

# Find shapefile
shp = next(extract_dir.rglob("*.shp"))

# Read and convert
gdf = gpd.read_file(shp)
gdf = gdf.rename(columns = {'NWS_REG':'id', 'NAME':'name'})
source_dict = {
    "source" : "NWS",
    "url" : url,
    "date_retreived" : {fetch_date},
}
gdf['properties'] = [source_dict] * len(gdf)
gdf[['id','name','geometry','properties']].to_parquet(parquet_path, index=False)

print(f"Saved GeoParquet: {parquet_path}")

# Cleanup: remove zip + extracted files
if zip_path.exists():
    zip_path.unlink()
if extract_dir.exists():
    shutil.rmtree(extract_dir)

In [ ]:
check = gpd.read_parquet(parquet_path)
check

## Get EPA Ecoregion Level 1 and 2 Boundaries

In [ ]:
fetch_date = datetime.now().strftime("%m-%d-%Y")

for level in ['l1', 'l2']:
    url = f"https://dmap-prod-oms-edc.s3.us-east-1.amazonaws.com/ORD/Ecoregions/cec_na/na_cec_eco_{level}.zip"
    parquet_path = Path(basedir, f"epa_ecoregion_{level}_geometry.all.parquet")

    extract_dir = Path(basedir, "tmp_extract")
    zip_path = Path(basedir, "temp.zip")

    # Download zip
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    zip_path.write_bytes(resp.content)

    # Extract
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)

    # Find shapefile
    shp = next(extract_dir.rglob("*.shp"))

    # Read and convert
    gdf = gpd.read_file(shp)

    gdf = gdf.rename(columns = {f'NA_{level.upper()}CODE':'id', f"NA_{level.upper()}NAME":'name'})
    source_dict = {
        "source" : "EPA",
        "url" : url,
        "date_retreived" : {fetch_date},
    }
    gdf['properties'] = [source_dict] * len(gdf)
    gdf[['id','name','geometry','properties']].to_parquet(parquet_path, index=False)

    print(f"Saved GeoParquet: {parquet_path}")

    # Cleanup: remove zip + extracted files
    if zip_path.exists():
        zip_path.unlink()
    if extract_dir.exists():
        shutil.rmtree(extract_dir)


In [ ]:
# view to check
for level in ['l1', 'l2']:
    parquet_path = Path(basedir, f"epa_ecoregion_{level}_geometry.all.parquet")
    display(gpd.read_parquet(parquet_path))

In [ ]:
## Copy into data/common/geometry on TEEHR hub
geom_dir = Path("/data/common/geometry")

files = [
    "rfc_geometry.all.parquet",
    "wfo_geometry.all.parquet",
    "nws_region_geometry.all.parquet",
    "epa_ecoregion_l1_geometry.all.parquet",
    "epa_ecoregion_l2_geometry.all.parquet"
]

for file in files:
    shutil.copy(Path(basedir, file), Path(geom_dir, file))

In [ ]:
## upload to s3://ciroh-rti-public-data/teehr-data-warehouse/common/geometry
for file in files:
    !aws s3 cp ./{file} s3://ciroh-rti-public-data/teehr-data-warehouse/common/geometry/{file} --acl public-read --profile default

### HUC12 polygons require far more processing - merging multiple subsets (by HUC2) and simplifying to a more managable size for mapping
- this processing has not yet been added below

In [ ]:
# update huc12s with properties
gdf = gpd.read_parquet('/data/common/geometry/huc12_geometry.all.parquet')
prop_dict = {
    "source" : "USGS",
    "url" : "https://prd-tnm.s3.amazonaws.com/index.html?prefix=StagedProducts/Hydrography/WBD/HU2/GPKG/",
    "date_retreived" : '05-10-2025',
}
gdf['properties'] = [prop_dict] * len(gdf)

In [ ]:
gdf.to_parquet('/data/common/geometry/huc12_geometry.all.parquet')

In [ ]:
!aws s3 cp /data/common/geometry/huc12_geometry.all.parquet s3://ciroh-rti-public-data/teehr-data-warehouse/common/geometry/huc12_geometry.all.parquet --acl public-read --profile default

In [ ]:
gdf